# yt-clips Premium Pipeline v2
## Colab T4 Setup — Studio-Grade Shorts

Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ════════════════════════════════════
# CELL 1: System + Python Dependencies
# ════════════════════════════════════
import sys, os, subprocess, json, yaml, time

print("Installing system deps...")
!apt-get update -qq && apt-get install -y -qq aria2 ffmpeg 

print("Installing Deno (bot bypass)...")
!curl -fsSL https://deno.land/x/install/install.sh | sh 
os.environ["PATH"] += ":/root/.deno/bin"

print("Installing Python deps...")
!pip install -q yt-dlp faster-whisper rich PyYAML opencv-python-headless numpy filterpy scipy 

print("Installing Premium GPU deps (YOLOv8 + PyTorch)...")
!pip install -q ultralytics torch --extra-index-url https://download.pytorch.org/whl/cu121 

print("Installing Premium Render deps (GFPGAN)...")
!pip install -q gfpgan basicsr 

# Verify GPU
gpu_ok = os.system('nvidia-smi ') == 0
print(f"GPU available: {gpu_ok}")
if not gpu_ok:
    print("WARNING: No GPU found! Set Runtime → T4 GPU and restart.")

print("\nALL DEPS INSTALLED ✅")

In [ ]:
# ════════════════════════════════════
# CELL 2: Upload your code files
# ════════════════════════════════════
# Method A: Drag & drop files
# Click the 📁 icon on left → drag all .py files + utils/ folder here
#
# Method B: Use your own git repo (uncomment & fill your repo URL)
# !git clone https://github.com/YOUR_USER/yt-clips.git /content/yt-clips
# %cd /content/yt-clips
#
# Method C: Upload via zip (uncomment)
# from google.colab import files
# uploaded = files.upload()  # upload yt-clips.zip
# !unzip -q yt-clips.zip -d /content/yt-clips
# %cd /content/yt-clips

print("Upload your Python files to /content/ using the 📁 sidebar")
print("Then come back and run the next cell.")

In [ ]:
# ════════════════════════════════════
# CELL 3: Create optimized config.yaml
# ════════════════════════════════════
import os
for folder in ['input', 'temp', 'transcripts', 'highlights', 'shorts', 'logs']:
    os.makedirs(folder, exist_ok=True)

config = """
paths:
  input:       input/
  temp:        temp/
  transcripts: transcripts/
  highlights:  highlights/
  shorts:      shorts/
  logs:        logs/

download:
  format: "bv*+ba/b"
  concurrent_fragments: 32
  use_aria2c: true
  output_filename: "video.mp4"
  po_token: ""
  proxy: ""

transcription:
  model: "small"
  language: "hi"
  device: "cuda"
  compute_type: "float16"

highlight:
  audio_energy_threshold: 0.3
  min_duration: 15
  max_duration: 20
  merge_gap: 8
  max_clips: 20
  fast_speech_wpm: 140
  silence_penalty_seconds: 1.5
  use_ai_refinement: true

premium:
  enabled: true
  face_enhancement: true
  frame_interpolation: true
  host_ref_photos: ""

layout:
  has_facecam: true
  facecam: {x: 0, y: 540, width: 320, height: 180}
  facecam_output_height: 400
  gameplay_output_height: 1520
  has_chat_overlay: true
  chat:
    side: "right"
    estimated_width: 350
    brightness_threshold: 30

export:
  width: 1080
  height: 1920
  fps: 60
  video_bitrate: "25M"
  audio_bitrate: "320k"
  crf: 18
  encoder: "h264_nvenc"
  enable_variable_speed: true
  crop_smooth_factor: 0.2

youtube:
  privacy_status: "private"
  category_id: "17"
  self_declared_made_for_kids: false
  shorts_max_seconds: 180
  upload_enabled: false
  schedule_interval_hours: 2
  niche: "Cricket"

ai:
  provider: "gemini"
  api_key: ""
  model: "gemini-2.0-flash-lite"
  image_model: "gemini-2.5-flash-image"

thumbnail:
  enabled: true
  use_ai: true
  template_path: "channel_logo.png"
  font_size: 120
  variants_count: 3

quality:
  black_threshold: 20
  silence_threshold_db: -35
  min_silence_duration: 1.0
  slow_wpm_threshold: 100
  frame_sample_count: 5

testing:
  enabled: false
  timeout_seconds: 120
  fail_fast: true

logging:
  level: "INFO"
  log_file: "logs/pipeline.log"
"""

with open('config.yaml', 'w') as f:
    f.write(config.strip())
print("config.yaml written ✅ premium.enabled=true")

In [ ]:
# ════════════════════════════════════
# CELL 4: Run Pre-Flight Tests
# ════════════════════════════════════
import subprocess
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-x", "--timeout=120", "-q"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-500:])
    print("\n❌ TESTS FAILED — fix before running pipeline")
else:
    print("\n✅ ALL TESTS PASSED — ready to pipeline")

In [ ]:
# ════════════════════════════════════
# CELL 5: RUN PIPELINE
# ════════════════════════════════════
YOUTUBE_URL = ""  # ← PASTE YOUR YOUTUBE VOD URL HERE

if not YOUTUBE_URL:
    print("❌ Paste a YouTube URL in the YOUTUBE_URL variable above")
else:
    !python pipeline.py "{YOUTUBE_URL}" --upload

In [ ]:
# ════════════════════════════════════
# CELL 6: Download Shorts from Colab
# ════════════════════════════════════
from google.colab import files
import zipfile, glob

shorts_dir = '/content/shorts'
if os.path.exists(shorts_dir):
    mp4s = glob.glob(f'{shorts_dir}/**/*.mp4', recursive=True)
    print(f"Found {len(mp4s)} shorts:")
    for mp4 in mp4s:
        print(f"  {mp4}")
    
    # Zip and download
    zip_path = '/content/shorts.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for mp4 in mp4s:
            zf.write(mp4, os.path.relpath(mp4, shorts_dir))
    files.download(zip_path)
else:
    print("No shorts/ directory found. Run the pipeline first.")